In [ ]:
import sys
from pathlib import Path
import polars as pl

sys.path.append(str(Path("..").resolve()))
from fleetsense.features.data_loader import get_dataset, FEATURES, get_features_and_target
from fleetsense.monitoring.distribution_monitoring import (
    monitor_all_features,
    build_baselines,
    check_drift,
    add_weighted_psi,
)
from fleetsense.monitoring.plotting import (
    plot_psi_heatmap,
    plot_psi_timeseries,
    rank_features_by_drift,
)
from fleetsense.model.base_model import load_baseline_model

In [ ]:
df = get_dataset()
df = df.with_columns(
    pl.col("week_start")
    .str.to_datetime("%Y-%m-%dT%H:%M:%S%.f", strict=False)  # skip if week_start is already a Date/Datetime
    .dt.truncate("1mo")
)

In [ ]:
model = load_baseline_model()
X_all, _ = get_features_and_target(df)
df = df.with_columns(pl.Series("predicted_ship_type", model.predict(X_all)))

In [ ]:
print(df.schema)

In [ ]:
from datetime import date

baselines = build_baselines(
    df.filter(pl.col("week_start").is_between(date(2025, 7, 1), date(2025, 8, 31), closed="left")),
    FEATURES,
    "ship_type",
    predicted_class_col="predicted_ship_type",
)

In [ ]:
results = monitor_all_features(
    baselines, df, FEATURES, "week_start", class_col="ship_type", predicted_class_col="predicted_ship_type"
)

weights = {
    "length_beam_ratio": 2.0,
    "draught_length_ratio": 1.2595515383,
    "width": 0.5517990614,
    "length": 0.4959847036,
    "min_draught": 0.293203546,
    "max_draught": 0.2179037024,
    "lat_mean": 0.2496088997,
    "sog_p90": 0.2000695289,
    "CargoY_ratio": 0.1561446202,
    "lon_mean": 0.3120285069,
    "max_speed": 0.1895011298,
    "draught_variability": 0.1895011298,
    "sog_median": 0.1234486355,
    "fishing_ratio": 0.1779419433,
    "CargoZ_ratio": 0.1052841995,
    "anchor_ratio": 0.1013210499,
    "moored_ratio": 0.2354076134,
    "lon_std": 0.2208760647,
    "time_span_seconds": 0.1013210499,
    "rot_std": 0.1138710238,
    "frac_time_slow": 0.152511733,
    "rot_mean_abs": 0.1227881106,
    "CargoOS_ratio": 0.1066052494,
    "n_pings": 0.1,
    "cog_variability": 0.1151920737,
    "mean_ping_interval_seconds": 0.1039631497,
    "lat_std": 0.1244394229,
    "mean_moving_speed": 0.1369893968,
    "sog_p10": 0.1072657744,
}
results = add_weighted_psi(results, weights=weights)

In [ ]:
results

In [ ]:
flagged = check_drift(results)
flagged.filter(pl.col("feature") == "__predicted_class_balance__")

In [ ]:
for row in flagged.iter_rows(named=True):
    print(f"{row['period']}: {row['feature']}, {row['ship_type']}, {row['psi']}")

In [ ]:
plot = plot_psi_heatmap(results, class_col="ship_type", figsize_per_panel=(10, 10))

In [ ]:
plot_psi_timeseries(results, "frac_time_slow", "ship_type")

In [ ]:
rank_features_by_drift(results, "ship_type")